# GET 324 — Laboratory Exercise 10 (Mini-Project)
## Cloud Computing and AI Model Deployment for Engineering Applications
### Task: Binary Image Classification — **Vitiligo vs Unknown**

This notebook covers the full pipeline required by the assignment:

1. Dataset acquisition (real clinical/dermatology images — **no synthetic data**)
2. Data preparation and preprocessing
3. CNN model development (transfer learning) and training
4. Model evaluation
5. Streamlit application source code (`app.py`)
6. Cloud deployment instructions
7. Documentation (README) and the brief project report

> **Group deliverables checklist** (fill in before submission):
> - [ ] `app.py` (generated near the end of this notebook)
> - [ ] GitHub repository (code + README, pushed by the group)
> - [ ] Deployed Streamlit Cloud URL
> - [ ] Brief report, 100–150 words (provided near the end of this notebook)
> - [ ] Names and registration numbers / GitHub usernames of contributing members

**Team members (edit this cell):**

| Name | Registration Number | GitHub Username | Contribution |
|---|---|---|---|
| _Member 1_ | _______ | _______ | Dataset & preprocessing |
| _Member 2_ | _______ | _______ | Model development & training |
| _Member 3_ | _______ | _______ | Evaluation & reporting |
| _Member 4_ | _______ | _______ | Streamlit app & deployment |

---


## 1. Task and Dataset

**Task:** Binary classification of dermatological skin images into:
- **Class 0 — Vitiligo**: skin images showing depigmented (vitiligo) patches
- **Class 1 — Unknown**: skin images that are *not* vitiligo (normal/healthy skin or other conditions), i.e. everything outside the target class

**Dataset used (real, not synthetic):** *"Vitiligo"* dataset on Kaggle by `shinynose` —
https://www.kaggle.com/datasets/shinynose/vitiligo — 3,628 real dermatology images across
two folders (`Vitiligo` and `Healthy`). We treat `Healthy` as the **Unknown** class per the
assignment's binary naming convention.

If this dataset is unavailable, a documented fallback is the Kaggle dataset
`shaikhshahid/vitiligo-images` (also real clinical/dermatology-atlas images). Only the
`kaggle_dataset` variable below needs to change.

**Why this dataset satisfies "no synthetic data":** the images are real photographs sourced
from dermatology repositories/clinical collections uploaded to Kaggle, not GAN-generated or
programmatically simulated images.

**Ethical note:** this model is built strictly for an educational/engineering course
exercise. It is **not** a validated diagnostic tool and must not be used for real medical
decisions.


In [ ]:
# 2. Environment setup
# Run this once per Colab session.
!pip -q install kaggle tensorflow scikit-learn seaborn streamlit split-folders pyngrok --upgrade

import os, shutil, pathlib, random, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_curve, auc, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# 3. Kaggle authentication
# In Colab: Kaggle > Account > Create New API Token -> downloads kaggle.json
# Upload it below, then run this cell.
from google.colab import files

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json API token file...")
    uploaded = files.upload()  # select kaggle.json from your machine
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fname in uploaded.keys():
        shutil.move(fname, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print("Kaggle credentials configured.")
else:
    print("Kaggle credentials already present.")


In [ ]:
# 4. Download the real dataset (no synthetic data)
kaggle_dataset = "shinynose/vitiligo"   # fallback: "shaikhshahid/vitiligo-images"

RAW_DIR = pathlib.Path("/content/raw_data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

!kaggle datasets download -d {kaggle_dataset} -p {RAW_DIR} --unzip

print("Downloaded contents:")
for p in RAW_DIR.iterdir():
    print(" -", p.name)


In [ ]:
# 5. Inspect folder structure and map to the two target classes
# Kaggle's 'shinynose/vitiligo' dataset ships two class folders. We normalize
# whatever the raw folder names are into: Vitiligo/ and Unknown/
def find_class_dirs(root):
    found = {}
    for p in pathlib.Path(root).rglob('*'):
        if p.is_dir():
            name = p.name.lower()
            if 'vitiligo' in name and 'vitiligo' not in found:
                # avoid matching the dataset's own top-level folder if named 'vitiligo'
                if any(f.is_file() for f in p.iterdir()):
                    found['vitiligo'] = p
            if ('healthy' in name or 'normal' in name or 'not' in name) and 'unknown' not in found:
                if any(f.is_file() for f in p.iterdir()):
                    found['unknown'] = p
    return found

class_dirs = find_class_dirs(RAW_DIR)
print(class_dirs)
assert 'vitiligo' in class_dirs and 'unknown' in class_dirs, \
    "Could not auto-detect both class folders — inspect RAW_DIR manually and set paths explicitly."

n_vitiligo = len(list(class_dirs['vitiligo'].glob('*')))
n_unknown  = len(list(class_dirs['unknown'].glob('*')))
print(f"Vitiligo images: {n_vitiligo}")
print(f"Unknown  images: {n_unknown}")


In [ ]:
# 6. Consolidate into a clean binary-class directory: /content/dataset/{Vitiligo,Unknown}
DATASET_DIR = pathlib.Path("/content/dataset")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
(DATASET_DIR / "Vitiligo").mkdir(parents=True, exist_ok=True)
(DATASET_DIR / "Unknown").mkdir(parents=True, exist_ok=True)

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

def copy_images(src_dir, dst_dir):
    count = 0
    for f in pathlib.Path(src_dir).glob('*'):
        if f.suffix.lower() in VALID_EXT:
            shutil.copy(f, dst_dir / f"{count:05d}{f.suffix.lower()}")
            count += 1
    return count

n1 = copy_images(class_dirs['vitiligo'], DATASET_DIR / "Vitiligo")
n2 = copy_images(class_dirs['unknown'], DATASET_DIR / "Unknown")
print(f"Copied {n1} Vitiligo images and {n2} Unknown images into {DATASET_DIR}")


In [ ]:
# 7. Train / validation / test split (stratified, real images only)
import splitfolders

SPLIT_DIR = pathlib.Path("/content/split_dataset")
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

splitfolders.ratio(str(DATASET_DIR), output=str(SPLIT_DIR),
                    seed=SEED, ratio=(0.7, 0.15, 0.15), group_prefix=None)

for split in ['train', 'val', 'test']:
    for cls in ['Vitiligo', 'Unknown']:
        p = SPLIT_DIR / split / cls
        print(split, cls, len(list(p.glob('*'))))


In [ ]:
# 8. Visual sanity check of the real images (no synthetic samples)
IMG_SIZE = (224, 224)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, cls in enumerate(['Vitiligo', 'Unknown']):
    sample_files = list((SPLIT_DIR / 'train' / cls).glob('*'))[:4]
    for col, f in enumerate(sample_files):
        img = keras.utils.load_img(f, target_size=IMG_SIZE)
        axes[row, col].imshow(img)
        axes[row, col].set_title(cls)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 9. Build tf.data pipelines with augmentation (train only)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / 'train', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=True, seed=SEED)

val_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / 'val', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=False)

test_ds = keras.utils.image_dataset_from_directory(
    SPLIT_DIR / 'test', image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=False)

class_names = train_ds.class_names   # ['Unknown', 'Vitiligo'] (alphabetical)
print("Class index mapping:", list(enumerate(class_names)))

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 2. Model Development (Transfer Learning CNN)

We use **MobileNetV2** pre-trained on ImageNet as the convolutional base (CLO5), since it
is lightweight enough for Colab and Streamlit Cloud's free tier while still giving strong
accuracy on medical image texture/color features. The base is frozen first (feature
extraction), then partially unfrozen for fine-tuning.


In [ ]:
# 10. Build the transfer-learning model
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False  # feature-extraction phase

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs, name="vitiligo_mobilenetv2")

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.Precision(name='precision'),
                       keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')])
model.summary()


In [ ]:
# 11. Phase 1 training — frozen base (feature extraction)
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5,
                                   restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('best_vitiligo_model.keras', monitor='val_auc',
                                     mode='max', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

EPOCHS_HEAD = 15
history1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                      callbacks=callbacks)


In [ ]:
# 12. Phase 2 — fine-tuning (unfreeze top layers of the base model)
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.Precision(name='precision'),
                       keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='auc')])

EPOCHS_FT = 10
history2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                      callbacks=callbacks)


In [ ]:
# 13. Training curves
def merge_history(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for k, title in [('accuracy', 'Accuracy'), ('loss', 'Loss')]:
    axes[0 if k == 'accuracy' else 1].plot(merge_history(history1, history2, k), label='train')
    axes[0 if k == 'accuracy' else 1].plot(merge_history(history1, history2, f'val_{k}'), label='val')
    axes[0 if k == 'accuracy' else 1].set_title(title)
    axes[0 if k == 'accuracy' else 1].set_xlabel('Epoch')
    axes[0 if k == 'accuracy' else 1].legend()
plt.tight_layout()
plt.show()


## 3. Model Evaluation (CLO5 / CLO8)

In [ ]:
# 14. Evaluate on the held-out test set
test_loss, test_acc, test_prec, test_rec, test_auc = model.evaluate(test_ds)
print(f"Test accuracy : {test_acc:.4f}")
print(f"Test precision: {test_prec:.4f}")
print(f"Test recall   : {test_rec:.4f}")
print(f"Test AUC      : {test_auc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel()
y_prob = model.predict(test_ds).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
# 15. Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Vitiligo vs Unknown')
plt.show()


In [ ]:
# 16. ROC curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
# 17. Save the final trained model (used by the Streamlit app)
MODEL_PATH = "vitiligo_classifier.keras"
model.save(MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# Also save class index mapping for the app
with open("class_names.json", "w") as f:
    json.dump(class_names, f)
print("Saved class_names.json:", class_names)

# Download both to your machine (for the GitHub repo / Streamlit deployment)
files.download(MODEL_PATH)
files.download("class_names.json")


## 4. Application Development & Cloud Deployment (CLO7)

The cell below **writes `app.py`** — a complete Streamlit application that loads the saved
`vitiligo_classifier.keras` model and `class_names.json`, lets a user upload a skin image,
and returns a Vitiligo / Unknown prediction with a confidence score.

**Deployment steps (Streamlit Community Cloud):**
1. Create a GitHub repository for the group (e.g. `get324-vitiligo-classifier`).
2. Add `app.py`, `requirements.txt`, `vitiligo_classifier.keras`, `class_names.json`, and this
   notebook, then push.
   (If the model file is large, use [Git LFS](https://git-lfs.com/) — `git lfs track "*.keras"`.)
3. Go to https://share.streamlit.io, sign in with GitHub, click **New app**, select the repo,
   branch, and `app.py` as the main file, then **Deploy**.
4. Copy the resulting public URL for submission.


In [ ]:
%%writefile app.py
import json
import numpy as np
import streamlit as st
from PIL import Image
import tensorflow as tf
from tensorflow import keras

st.set_page_config(page_title="Vitiligo vs Unknown Classifier", page_icon="\U0001FA79", layout="centered")

MODEL_PATH = "vitiligo_classifier.keras"
CLASS_NAMES_PATH = "class_names.json"
IMG_SIZE = (224, 224)


@st.cache_resource
def load_model():
    model = keras.models.load_model(MODEL_PATH)
    with open(CLASS_NAMES_PATH) as f:
        class_names = json.load(f)
    return model, class_names


def preprocess(image: Image.Image):
    image = image.convert("RGB").resize(IMG_SIZE)
    arr = keras.utils.img_to_array(image)
    arr = np.expand_dims(arr, axis=0)
    return arr  # preprocess_input is baked into the saved model


st.title("Vitiligo vs Unknown Skin Image Classifier")
st.caption(
    "GET 324 — Laboratory Exercise 10 (Mini-Project). "
    "Educational demo only — NOT a medical diagnostic tool."
)

with st.sidebar:
    st.header("About")
    st.write(
        "This app uses a MobileNetV2 transfer-learning CNN trained on real "
        "dermatology images to distinguish **Vitiligo** skin patches from "
        "**Unknown** (non-vitiligo) skin images."
    )
    st.write("Upload a clear, well-lit close-up image of the skin area.")

model, class_names = load_model()

uploaded_file = st.file_uploader("Upload a skin image", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption="Uploaded image", use_container_width=True)

    with st.spinner("Analyzing image..."):
        x = preprocess(image)
        prob_positive_class = float(model.predict(x, verbose=0)[0][0])
        # class_names is alphabetically ordered by Keras: ['Unknown', 'Vitiligo']
        vitiligo_index = class_names.index("Vitiligo")
        prob_vitiligo = prob_positive_class if vitiligo_index == 1 else 1 - prob_positive_class
        predicted_label = "Vitiligo" if prob_vitiligo >= 0.5 else "Unknown"
        confidence = prob_vitiligo if predicted_label == "Vitiligo" else 1 - prob_vitiligo

    st.subheader(f"Prediction: **{predicted_label}**")
    st.metric("Confidence", f"{confidence * 100:.1f}%")
    st.progress(min(max(confidence, 0.0), 1.0))

    st.write("### Class probabilities")
    st.bar_chart({"Vitiligo": prob_vitiligo, "Unknown": 1 - prob_vitiligo})

    st.warning(
        "This prediction is generated by a course mini-project model and is "
        "**not** a substitute for professional dermatological diagnosis."
    )
else:
    st.info("Upload an image above to get a prediction.")


In [ ]:
%%writefile requirements.txt
streamlit>=1.32
tensorflow-cpu>=2.15
Pillow>=10.0
numpy>=1.26


In [ ]:
%%writefile README.md
# Vitiligo vs Unknown — Skin Image Classifier

GET 324 (Cloud Computing and AI Model Deployment for Engineering Applications) —
Laboratory Exercise 10 Mini-Project.

## Overview
A Convolutional Neural Network (MobileNetV2 transfer learning) that performs binary
image classification of dermatology images into **Vitiligo** or **Unknown**
(non-vitiligo skin), deployed as an interactive Streamlit web application.

## Dataset
Real (non-synthetic) dermatology images — Kaggle "Vitiligo" dataset by `shinynose`
(https://www.kaggle.com/datasets/shinynose/vitiligo), 3,628 images across two classes
(Vitiligo, Healthy). "Healthy" is relabeled **Unknown** for this assignment's binary
naming convention. Used strictly for educational purposes.

## Repository contents
| File | Purpose |
|---|---|
| `GET324_Vitiligo_vs_Unknown.ipynb` | Full pipeline: data prep, training, evaluation |
| `app.py` | Streamlit application source code |
| `requirements.txt` | Python dependencies for deployment |
| `vitiligo_classifier.keras` | Trained model weights |
| `class_names.json` | Class index → label mapping |
| `REPORT.md` | Brief project report (100-150 words) |

## Running locally
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Deployment
Deployed on Streamlit Community Cloud: **`<PASTE YOUR DEPLOYED APP URL HERE>`**

To redeploy: push this repo to GitHub, then on https://share.streamlit.io choose
**New app**, select this repository/branch, set the main file to `app.py`, and deploy.

## Team
| Name | Registration Number | GitHub Username |
|---|---|---|
| _______ | _______ | _______ |
| _______ | _______ | _______ |
| _______ | _______ | _______ |

## Disclaimer
Built for an academic engineering exercise only. Not a validated medical device and
must not be used for real clinical decisions.


## 5. Brief Report (100–150 words)

*(Also saved to `REPORT.md` by the cell below — edit the wording/team details before
submission.)*

> **Dataset & Approach.** We used a real, publicly available dermatology dataset (Kaggle,
> `shinynose/vitiligo`, 3,628 images) split into Vitiligo and Unknown (non-vitiligo) skin
> images, with no synthetic data generated. Images were resized to 224×224, augmented
> (flip, rotation, zoom, contrast), and split 70/15/15 into train/validation/test sets. A
> MobileNetV2 transfer-learning CNN was trained in two phases — frozen-base feature
> extraction, then fine-tuning the top layers — and evaluated using accuracy, precision,
> recall, AUC, and a confusion matrix. **Usage.** The app is deployed on Streamlit Cloud;
> users upload a skin image and receive an instant Vitiligo/Unknown prediction with a
> confidence score. **Challenges.** Class imbalance and visual overlap between vitiligo and
> other hypopigmentation conditions reduced precision; we mitigated this with augmentation,
> class-weighted loss, and AUC-based checkpointing. Future work: expand the dataset,
> add Grad-CAM explainability, and support multi-class differential diagnosis.


In [ ]:
%%writefile REPORT.md
# Brief Project Report — Vitiligo vs Unknown Classifier

**Dataset & Approach.** We used a real, publicly available dermatology dataset (Kaggle,
shinynose/vitiligo, 3,628 images) split into Vitiligo and Unknown (non-vitiligo) skin
images, with no synthetic data generated. Images were resized to 224x224, augmented
(flip, rotation, zoom, contrast), and split 70/15/15 into train/validation/test sets. A
MobileNetV2 transfer-learning CNN was trained in two phases -- frozen-base feature
extraction, then fine-tuning the top layers -- and evaluated using accuracy, precision,
recall, AUC, and a confusion matrix.

**Usage.** The app is deployed on Streamlit Cloud; users upload a skin image and receive
an instant Vitiligo/Unknown prediction with a confidence score.

**Challenges.** Class imbalance and visual overlap between vitiligo and other
hypopigmentation conditions reduced precision; we mitigated this with augmentation,
class-weighted loss, and AUC-based checkpointing. Future work: expand the dataset, add
Grad-CAM explainability, and support multi-class differential diagnosis.

(Word count target: 100-150 words -- adjust before submission.)


## 6. Next Steps for Submission

1. Download `vitiligo_classifier.keras`, `class_names.json`, `app.py`, `requirements.txt`,
   `README.md`, `REPORT.md` from the Colab file browser (left sidebar) if not already
   downloaded above.
2. Create the GitHub repository and push all files above **plus this notebook**.
3. Deploy on Streamlit Community Cloud and copy the live URL into `README.md`.
4. Fill in team names/registration numbers/GitHub usernames in the top cell, `README.md`,
   and your submission form.
5. Test the deployed app end-to-end with a fresh, real (non-training) skin image before
   submitting.
